# Colab: local LLM -> JSON -> validator -> KOMPAS macro

Этот ноутбук рассчитан на Google Colab с GPU T4.

Что делает:

1. Клонирует проект с GitHub.
2. Ставит `llama-cpp-python` и `huggingface_hub`.
3. Скачивает GGUF-модель Qwen Coder с Hugging Face.
4. Генерирует JSON по тексту пользователя.
5. Проверяет JSON через `resolve_placements()` и `validate_plan()`.
6. Если есть ошибка, возвращает ошибку модели и просит исправить JSON.
7. Сохраняет JSON, `.m3m` и журнал попыток в Google Drive.

## 1. Проверка GPU

В Colab включи: `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
!nvidia-smi

## 2. Google Drive для логов и результатов

In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/json-cad-pipeline')
DRIVE_LOG_DIR = DRIVE_ROOT / 'logs'
DRIVE_JSON_DIR = DRIVE_ROOT / 'generated_json'
DRIVE_MACRO_DIR = DRIVE_ROOT / 'generated_macros'
LOCAL_MODEL_DIR = Path('/content/models')

for path in [DRIVE_LOG_DIR, DRIVE_JSON_DIR, DRIVE_MACRO_DIR, LOCAL_MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)

## 3. Клонирование проекта с GitHub

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/12Jun34/json-cad-pipeline.git'
PROJECT_ROOT = Path('/content/json-cad-pipeline')

if PROJECT_ROOT.exists():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)

print('Project root:', PROJECT_ROOT)

## 4. Установка зависимостей

Эта ячейка ставит готовый CUDA wheel для `llama-cpp-python`. Это обычно занимает минуты, а не десятки минут. Если wheel для текущей CUDA временно недоступен, смотри fallback-команду в комментарии внутри ячейки.

In [ ]:
%pip install -q --upgrade huggingface_hub
%pip install -q --upgrade llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

# Slow fallback only if the prebuilt CUDA wheel above fails:
# !CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q --upgrade --force-reinstall --no-cache-dir llama-cpp-python

## 5. Скачивание модели

По умолчанию берется `Qwen/Qwen2.5-Coder-14B-Instruct-GGUF` и квант `Q4_K_M`. Для T4 это хороший стартовый вариант.

Если будет нехватка памяти или слишком медленно, поменяй `MODEL_REPO` на `Qwen/Qwen2.5-Coder-7B-Instruct-GGUF`.

In [ ]:
from huggingface_hub import list_repo_files, snapshot_download

MODEL_REPO = 'Qwen/Qwen2.5-Coder-14B-Instruct-GGUF'
PREFERRED_QUANT = 'q4_k_m'

repo_files = list_repo_files(MODEL_REPO)
gguf_files = [name for name in repo_files if name.lower().endswith('.gguf')]
candidates = [name for name in gguf_files if PREFERRED_QUANT in name.lower()]

if not candidates:
    raise RuntimeError(f'No {PREFERRED_QUANT} GGUF file found in {MODEL_REPO}. Available: {gguf_files[:10]}')

MODEL_FILE = sorted(candidates)[0]
MODEL_DIR = Path(snapshot_download(
    repo_id=MODEL_REPO,
    allow_patterns=[f'*{PREFERRED_QUANT}*.gguf', f'*{PREFERRED_QUANT.upper()}*.gguf'],
    local_dir=str(LOCAL_MODEL_DIR),
))
MODEL_PATH = MODEL_DIR / MODEL_FILE

print('Model repo:', MODEL_REPO)
print('Model file:', MODEL_FILE)
print('Downloaded GGUF files:', [path.name for path in MODEL_DIR.glob('*.gguf')])
print('Model path:', MODEL_PATH)

## 6. Импорт CAD-пайплайна

In [ ]:
import csv
import json
import re
import sys
from datetime import datetime

SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from macro_generator import write_macro
from placement_resolver import resolve_placements
from validator import validate_plan

LOG_PATH = DRIVE_LOG_DIR / 'json_repair_log.jsonl'
CANDIDATE_LOG_PATH = DRIVE_LOG_DIR / 'candidate_log.jsonl'
CANDIDATE_TABLE_PATH = DRIVE_LOG_DIR / 'candidate_table.csv'
ACCEPTED_TABLE_PATH = DRIVE_LOG_DIR / 'accepted_candidates.csv'
REJECTED_TABLE_PATH = DRIVE_LOG_DIR / 'rejected_candidates.csv'
TEST_SUMMARY_TABLE_PATH = DRIVE_LOG_DIR / 'test_summary.csv'
SFT_PATH = DRIVE_LOG_DIR / 'sft_dataset.jsonl'
TEST_REQUESTS_PATH = PROJECT_ROOT / 'ml_json_generator' / 'test_requests.jsonl'

print('LOG_PATH:', LOG_PATH)
print('CANDIDATE_TABLE_PATH:', CANDIDATE_TABLE_PATH)
print('TEST_REQUESTS_PATH:', TEST_REQUESTS_PATH)

## 7. Загрузка локальной модели

In [ ]:
from llama_cpp import Llama

N_CTX = 8192
N_GPU_LAYERS = -1
TEMPERATURE = 0.15
CANDIDATE_COUNT = 8
CANDIDATE_TEMPERATURES = [0.15, 0.20, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75]
MAX_TOKENS = 2500
MAX_REPAIR_ATTEMPTS = 3

llm = Llama(
    model_path=str(MODEL_PATH),
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    verbose=False,
)

def chat(messages, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    result = llm.create_chat_completion(
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return result['choices'][0]['message']['content']

## 8. Спецификация JSON для модели

In [ ]:
CAD_JSON_SPEC = '''
You generate JSON plans for a KOMPAS macro generator.
Return only valid JSON. Do not use Markdown. Do not explain anything.

Top-level object:
{
  "version": "0.1",
  "units": "mm",
  "part_name": "ascii_snake_case_name",
  "commands": [ ... ]
}

Allowed command types only:
- create_box
- cut_box
- create_prism
- cut_prism
- create_triangle
- cut_triangle
- create_cylinder
- cut_cylinder

Never create object-specific command types. Forbidden examples:
- create_birdhouse
- create_plate_with_holes
- create_screwdriver
- create_table

Semantic placed box:
{
  "id": "body",
  "type": "create_box",
  "size": [width_y, height_z, depth_x],
  "placement": {"origin": [x, y, z]}
}

Triangle prism attached on top of a box:
{
  "id": "roof",
  "type": "create_triangle",
  "size": [width_y, height_z, depth_x],
  "attach": {"target": "body", "face": "top", "position": "center"}
}

Cylinder or cylindrical cut attached to front face:
{
  "id": "hole",
  "type": "cut_cylinder",
  "radius": 10,
  "depth": "half",
  "attach": {
    "target": "body",
    "face": "front",
    "position": "center",
    "offset": [dy, dz]
  }
}

Extrude modes:
- normal
- reverse
- middle

Rules:
- Use millimeters.
- Use stable ASCII ids.
- Every command id must be unique.
- A command can attach only to an object created earlier.
- Prefer placement/attach when the user describes relative positioning.
- Use low-level coordinates when the user gives exact coordinates.
- For a protruding peg/perch on the front face, use create_cylinder with extrude="reverse".
- For an inward hole on the front face, use cut_cylinder with extrude="normal".
'''.strip()

## 9. Примеры и память удачных генераций

In [ ]:
SEED_EXAMPLE_PATHS = [
    PROJECT_ROOT / 'examples' / 'birdhouse_attach.json',
    PROJECT_ROOT / 'examples' / 'plate_four_holes.json',
    PROJECT_ROOT / 'examples' / 'primitives_demo.json',
]

def load_json_file(path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def load_seed_examples():
    examples = []
    for path in SEED_EXAMPLE_PATHS:
        data = load_json_file(path)
        if data is not None:
            examples.append(data)
    return examples

def load_memory_examples(limit=3):
    if not LOG_PATH.exists():
        return []
    records = []
    for line in LOG_PATH.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        record = json.loads(line)
        if record.get('success') and record.get('final_plan'):
            records.append(record['final_plan'])
    return records[-limit:]

def format_examples(examples, limit=4):
    if not examples:
        return 'No examples yet.'
    chunks = []
    for index, example in enumerate(examples[-limit:], start=1):
        chunks.append(f'Example {index}:\n' + json.dumps(example, ensure_ascii=False, indent=2))
    return '\n\n'.join(chunks)

## 10. Проверка JSON и сохранение результата

In [ ]:
def extract_json_object(text):
    cleaned = text.strip()
    fence_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', cleaned, flags=re.DOTALL)
    if fence_match:
        cleaned = fence_match.group(1)
    first = cleaned.find('{')
    last = cleaned.rfind('}')
    if first == -1 or last == -1 or last <= first:
        raise ValueError('Model output does not contain a JSON object')
    return json.loads(cleaned[first:last + 1])

def validate_and_resolve(plan):
    resolved_plan, was_resolved = resolve_placements(plan)
    validate_plan(resolved_plan)
    return resolved_plan, was_resolved

def save_generated_outputs(raw_plan, resolved_plan):
    part_name = resolved_plan['part_name']
    raw_json_path = DRIVE_JSON_DIR / f'{part_name}.json'
    resolved_json_path = DRIVE_JSON_DIR / f'{part_name}.resolved.json'
    raw_json_path.write_text(json.dumps(raw_plan, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    resolved_json_path.write_text(json.dumps(resolved_plan, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    macro_path = write_macro(resolved_plan, DRIVE_MACRO_DIR)
    return {
        'raw_json_path': str(raw_json_path),
        'resolved_json_path': str(resolved_json_path),
        'macro_path': str(macro_path),
    }

## 11. Промпты генерации и ремонта

In [ ]:
def build_system_message():
    return (
        'You are a CAD JSON planner. '
        'You output only valid JSON for the provided schema. '
        'You never invent object-specific commands.'
    )

def build_generation_messages(user_request, part_name):
    examples = load_seed_examples() + load_memory_examples(limit=3)
    user_message = f'''
{CAD_JSON_SPEC}

Known good examples:
{format_examples(examples, limit=4)}

User request:
{user_request}

Use this part_name: {part_name}

Return only the JSON plan.
'''.strip()
    return [
        {'role': 'system', 'content': build_system_message()},
        {'role': 'user', 'content': user_message},
    ]

def build_repair_messages(user_request, part_name, bad_output, error):
    user_message = f'''
{CAD_JSON_SPEC}

Original user request:
{user_request}

Required part_name: {part_name}

Your previous output was invalid:
{bad_output}

Validator error:
{error}

Fix the JSON. Return only one complete valid JSON object.
'''.strip()
    return [
        {'role': 'system', 'content': build_system_message()},
        {'role': 'user', 'content': user_message},
    ]

## 12. Цикл: генерация -> ошибка -> исправление

In [ ]:
TABLE_FIELDS = [
    'created_at',
    'test_id',
    'batch_index',
    'user_request',
    'part_name',
    'candidate_index',
    'temperature',
    'verdict',
    'accepted',
    'duplicate_concept',
    'concept_signature',
    'total_score',
    'validity_score',
    'geometry_score',
    'simplicity_score',
    'style_score',
    'error_stage',
    'error_message',
    'error_description',
    'raw_output',
    'raw_plan_json',
    'resolved_plan_json',
]

SUMMARY_FIELDS = [
    'created_at',
    'test_id',
    'batch_index',
    'user_request',
    'part_name',
    'candidate_count',
    'valid_count',
    'invalid_count',
    'accepted_count',
    'best_score',
    'best_concept_signature',
    'success',
]

def append_jsonl(path, record):
    with path.open('a', encoding='utf-8') as file:
        file.write(json.dumps(record, ensure_ascii=False) + '\n')

def append_log(record):
    append_jsonl(LOG_PATH, record)

def _table_value(value):
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    if value is None:
        return ''
    return value

def append_candidate_table(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists()
    row = {field: _table_value(record.get(field)) for field in TABLE_FIELDS}
    with path.open('a', encoding='utf-8-sig', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=TABLE_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)

def append_summary_table(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists()
    row = {field: _table_value(record.get(field)) for field in SUMMARY_FIELDS}
    with path.open('a', encoding='utf-8-sig', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=SUMMARY_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)

def error_description(stage, message):
    if stage == 'extract_json':
        return 'Model did not return a clean JSON object. Check Markdown fences, comments, or missing braces.'
    if stage == 'resolve_or_validate':
        return 'JSON was parsed, but placement resolving or CAD schema validation failed. Check command types, sizes, ids, attach targets, and geometry fields.'
    if stage == 'save_outputs':
        return 'Plan was valid, but saving JSON or macro failed. Check part_name and Drive paths.'
    return f'Unexpected error: {message}'

def concept_signature(plan):
    if not plan or not isinstance(plan.get('commands'), list):
        return 'no_plan'
    parts = []
    for command in plan['commands']:
        command_type = command.get('type', 'unknown')
        if 'attach' in command:
            attach = command.get('attach') or {}
            mode = f"attach:{attach.get('face', 'unknown')}:{attach.get('position', 'center')}"
        elif 'placement' in command:
            mode = 'placement'
        else:
            mode = 'low_level'
        parts.append(f'{command_type}:{mode}')
    return '|'.join(parts)

def score_candidate(raw_plan, resolved_plan, was_resolved):
    commands = raw_plan.get('commands', []) if isinstance(raw_plan, dict) else []
    command_count = len(commands)
    attach_count = sum(1 for command in commands if 'attach' in command)
    placement_count = sum(1 for command in commands if 'placement' in command)
    low_level_count = command_count - attach_count - placement_count
    readable_ids = sum(1 for command in commands if isinstance(command.get('id'), str) and command['id'].replace('_', '').isalnum())

    validity = 40
    geometry = 20
    simplicity = max(0, 20 - max(0, command_count - 4) * 3 - low_level_count)
    style = min(10, readable_ids * 2) + min(10, (attach_count + placement_count) * 2)
    if was_resolved:
        geometry += 5
    total = validity + geometry + simplicity + style
    return {
        'total': min(100, total),
        'validity': validity,
        'geometry': min(25, geometry),
        'simplicity': simplicity,
        'style': style,
        'command_count': command_count,
        'attach_count': attach_count,
        'placement_count': placement_count,
    }

def evaluate_candidate(user_request, part_name, candidate_index, temperature, output, test_id=None, batch_index=None):
    created_at = datetime.now().isoformat(timespec='seconds')
    base = {
        'created_at': created_at,
        'test_id': test_id,
        'batch_index': batch_index,
        'user_request': user_request,
        'part_name': part_name,
        'candidate_index': candidate_index,
        'temperature': temperature,
        'raw_output': output,
        'accepted': False,
        'duplicate_concept': False,
    }

    try:
        raw_plan = extract_json_object(output)
    except Exception as error:
        message = str(error)
        return {
            **base,
            'verdict': 'invalid',
            'error_stage': 'extract_json',
            'error_message': message,
            'error_description': error_description('extract_json', message),
        }

    try:
        resolved_plan, was_resolved = validate_and_resolve(raw_plan)
    except Exception as error:
        message = str(error)
        return {
            **base,
            'verdict': 'invalid',
            'error_stage': 'resolve_or_validate',
            'error_message': message,
            'error_description': error_description('resolve_or_validate', message),
            'raw_plan_json': raw_plan,
            'concept_signature': concept_signature(raw_plan),
        }

    score = score_candidate(raw_plan, resolved_plan, was_resolved)
    return {
        **base,
        'verdict': 'valid',
        'raw_plan': raw_plan,
        'resolved_plan': resolved_plan,
        'raw_plan_json': raw_plan,
        'resolved_plan_json': resolved_plan,
        'concept_signature': concept_signature(raw_plan),
        'score': score,
        'total_score': score['total'],
        'validity_score': score['validity'],
        'geometry_score': score['geometry'],
        'simplicity_score': score['simplicity'],
        'style_score': score['style'],
    }

def select_best_conceptual_candidates(candidates, max_selected=3):
    best_by_concept = {}
    for candidate in candidates:
        if candidate.get('verdict') != 'valid':
            continue
        signature = candidate.get('concept_signature')
        current_best = best_by_concept.get(signature)
        if current_best is None or candidate.get('total_score', 0) > current_best.get('total_score', 0):
            best_by_concept[signature] = candidate

    selected_ids = {id(candidate) for candidate in sorted(best_by_concept.values(), key=lambda item: item.get('total_score', 0), reverse=True)[:max_selected]}
    for candidate in candidates:
        if candidate.get('verdict') != 'valid':
            continue
        candidate['accepted'] = id(candidate) in selected_ids
        best_for_concept = best_by_concept.get(candidate.get('concept_signature'))
        candidate['duplicate_concept'] = best_for_concept is not None and id(candidate) != id(best_for_concept)
    return [candidate for candidate in candidates if candidate.get('accepted')]

def write_candidate_logs(candidates):
    for candidate in candidates:
        serializable = {key: value for key, value in candidate.items() if key not in {'raw_plan', 'resolved_plan'}}
        append_jsonl(CANDIDATE_LOG_PATH, serializable)
        append_candidate_table(CANDIDATE_TABLE_PATH, serializable)
        if candidate.get('accepted'):
            append_candidate_table(ACCEPTED_TABLE_PATH, serializable)
        if candidate.get('verdict') != 'valid':
            append_candidate_table(REJECTED_TABLE_PATH, serializable)

def load_test_requests(path=TEST_REQUESTS_PATH, limit=None):
    requests = []
    with Path(path).open('r', encoding='utf-8') as file:
        for line_index, line in enumerate(file, start=1):
            if not line.strip():
                continue
            item = json.loads(line)
            test_id = item.get('id') or f'test_{line_index:03d}'
            user_request = item.get('request') or item.get('user_request')
            if not user_request:
                raise ValueError(f'Test request {test_id} has no request field')
            part_name = item.get('part_name') or test_id
            requests.append({'id': test_id, 'request': user_request, 'part_name': part_name})
            if limit is not None and len(requests) >= limit:
                break
    return requests

def generate_candidate_tournament(user_request, part_name, candidate_count=CANDIDATE_COUNT, test_id=None, batch_index=None):
    messages = build_generation_messages(user_request, part_name)
    candidates = []

    for candidate_index in range(1, candidate_count + 1):
        temperature = CANDIDATE_TEMPERATURES[(candidate_index - 1) % len(CANDIDATE_TEMPERATURES)]
        output = chat(messages, temperature=temperature)
        candidate = evaluate_candidate(user_request, part_name, candidate_index, temperature, output, test_id=test_id, batch_index=batch_index)
        candidates.append(candidate)

    accepted = select_best_conceptual_candidates(candidates)
    for candidate in accepted:
        try:
            candidate['paths'] = save_generated_outputs(candidate['raw_plan'], candidate['resolved_plan'])
        except Exception as error:
            candidate['accepted'] = False
            candidate['verdict'] = 'invalid'
            candidate['error_stage'] = 'save_outputs'
            candidate['error_message'] = str(error)
            candidate['error_description'] = error_description('save_outputs', str(error))

    write_candidate_logs(candidates)

    record = {
        'created_at': datetime.now().isoformat(timespec='seconds'),
        'test_id': test_id,
        'batch_index': batch_index,
        'success': bool([candidate for candidate in candidates if candidate.get('accepted')]),
        'user_request': user_request,
        'part_name': part_name,
        'candidate_count': candidate_count,
        'valid_count': sum(1 for candidate in candidates if candidate.get('verdict') == 'valid'),
        'invalid_count': sum(1 for candidate in candidates if candidate.get('verdict') != 'valid'),
        'accepted_count': sum(1 for candidate in candidates if candidate.get('accepted')),
        'accepted_candidates': [candidate for candidate in candidates if candidate.get('accepted')],
        'rejected_candidates': [candidate for candidate in candidates if candidate.get('verdict') != 'valid'],
        'all_candidates': [{key: value for key, value in candidate.items() if key not in {'raw_plan', 'resolved_plan'}} for candidate in candidates],
    }
    accepted_candidates = [candidate for candidate in candidates if candidate.get('accepted')]
    best_candidate = max(accepted_candidates, key=lambda item: item.get('total_score', 0), default=None)
    record['best_score'] = best_candidate.get('total_score') if best_candidate else None
    record['best_concept_signature'] = best_candidate.get('concept_signature') if best_candidate else None
    append_log(record)
    append_summary_table(TEST_SUMMARY_TABLE_PATH, record)
    return record

def generate_plan_with_repair(user_request, part_name, max_attempts=MAX_REPAIR_ATTEMPTS):
    return generate_candidate_tournament(user_request, part_name, candidate_count=CANDIDATE_COUNT)

def run_test_file(path=TEST_REQUESTS_PATH, limit=None, candidate_count=CANDIDATE_COUNT):
    tests = load_test_requests(path, limit=limit)
    records = []
    for batch_index, test in enumerate(tests, start=1):
        print(f"[{batch_index}/{len(tests)}] {test['id']}: {test['request']}")
        record = generate_candidate_tournament(
            test['request'],
            test['part_name'],
            candidate_count=candidate_count,
            test_id=test['id'],
            batch_index=batch_index,
        )
        records.append(record)
        print(f"  valid={record['valid_count']} invalid={record['invalid_count']} accepted={record['accepted_count']} best={record.get('best_score')}")
    return records

## 13. Тестовый запуск

In [ ]:
# Fast smoke test: run only the first 2 tasks with 2 candidates each.
# Full dataset run: set TEST_LIMIT = None and TEST_CANDIDATES = 8.
TEST_LIMIT = 2
TEST_CANDIDATES = 2

records = run_test_file(TEST_REQUESTS_PATH, limit=TEST_LIMIT, candidate_count=TEST_CANDIDATES)
print(json.dumps({
    'tests_run': len(records),
    'successful_tests': sum(1 for record in records if record.get('success')),
    'candidate_table': str(CANDIDATE_TABLE_PATH),
    'accepted_table': str(ACCEPTED_TABLE_PATH),
    'rejected_table': str(REJECTED_TABLE_PATH),
    'summary_table': str(TEST_SUMMARY_TABLE_PATH),
}, ensure_ascii=False, indent=2))

## 14. SFT-датасет для будущего дообучения

Эта ячейка превращает успешные записи из лога в `jsonl`-датасет формата chat messages.

In [ ]:
def build_sft_dataset():
    if not LOG_PATH.exists():
        return 0

    count = 0
    with LOG_PATH.open('r', encoding='utf-8') as source, SFT_PATH.open('w', encoding='utf-8') as target:
        for line in source:
            if not line.strip():
                continue
            record = json.loads(line)
            if not record.get('success'):
                continue

            accepted_plans = []
            for candidate in record.get('accepted_candidates', []):
                plan = candidate.get('raw_plan_json') or candidate.get('raw_plan')
                if plan:
                    accepted_plans.append(plan)
            if record.get('final_plan'):
                accepted_plans.append(record['final_plan'])

            for plan in accepted_plans:
                sample = {
                    'messages': [
                        {'role': 'system', 'content': build_system_message()},
                        {'role': 'user', 'content': CAD_JSON_SPEC + '\n\nUser request:\n' + record['user_request']},
                        {'role': 'assistant', 'content': json.dumps(plan, ensure_ascii=False)},
                    ]
                }
                target.write(json.dumps(sample, ensure_ascii=False) + '\n')
                count += 1
    return count

count = build_sft_dataset()
print(f'SFT samples written: {count} -> {SFT_PATH}')